In [1]:
# PFE Renault Tanger — Pipeline ETL
## Étape 1 : Extraction, Nettoyage, Transformation des données eau

In [2]:
import pandas as pd
import numpy as np
import warnings
import os
warnings.filterwarnings('ignore')

print("Librairies chargées ✓")

Librairies chargées ✓


In [3]:
import os
print(os.getcwd())


/Users/akrambelhaj/Desktop/PFE_Renault/notebooks


In [4]:
FICHIER_EXCEL  = "../data/Synthèse Eaux 2026 VF (1).xlsm"
FEUILLE        = "database_Eaux"
FICHIER_SORTIE = "../outputs/dataset_eau_propre.csv"
SEUIL_OBJECTIF = 1.25   # m³/véhicule
TCM_MIN_PROD   = 100    # véhicules minimum

In [5]:
df = pd.read_excel(FICHIER_EXCEL, sheet_name=FEUILLE, header=0)
df.columns = df.columns.str.strip()
df = df[df['TCM'].notna()].copy()
df = df.sort_values('Date').reset_index(drop=True)

print(f"Lignes chargées  : {len(df)}")
print(f"Période          : {df['Date'].min().date()} → {df['Date'].max().date()}")
df.head()

Lignes chargées  : 112
Période          : 2026-01-01 → 2026-04-22


,Date,KPI EAU,TCM,E_Appro Facturation,EI_Facturation,EP_Facturation,E. Appro Looker,EI_Looker,EP_Looker,ED_Total,...,EI_Incendie,EI Autres Process,ED_PB,ED_AEB,ED_TC,ED_BD,ED_UE,EOR TAR UE,EOR TAR BD,EOR TAR ST
0,2026-01-01,NaN,0.0,724.0,614.0,89.0,590.0,590.0,0.0,450.0,...,113.0,88.0,0.0,28.0,0.0,2.0,10.0,27.0,61.0,0.0
1,2026-01-02,NaN,0.0,784.0,681.0,120.0,660.0,608.0,52.0,220.0,...,156.0,85.0,0.0,15.0,0.0,8.0,0.0,44.0,14.0,2.0
2,2026-01-03,NaN,0.0,783.0,580.0,140.0,732.0,590.0,142.0,312.0,...,81.0,87.0,4.0,71.0,4.0,9.0,0.0,55.0,24.0,1.0
3,2026-01-04,NaN,0.0,787.0,705.0,80.0,759.0,683.0,76.0,100.0,...,36.0,294.0,0.0,3.0,0.0,18.0,0.0,24.0,59.0,0.0
4,2026-01-05,24.022222,45.0,1081.0,780.0,254.0,1103.0,829.0,274.0,574.0,...,15.0,92.0,8.0,63.0,8.0,61.0,0.0,59.0,5.0,3.0


In [6]:
df['is_arret']   = (df['TCM'] < TCM_MIN_PROD).astype(int)
df['is_weekend'] = (df['Date'].dt.dayofweek >= 5).astype(int)
df['is_lundi']   = (df['Date'].dt.dayofweek == 0).astype(int)

cols_conso = ['ED_Total', 'EOR_Total', 'EI_Looker', 'EP_Looker']
for col in cols_conso:
    nb = (df[col] == 0).sum()
    if nb > 0:
        df[col] = df[col].replace(0, np.nan)
        print(f"  {col} : {nb} zéro(s) → NaN")

  ED_Total : 1 zéro(s) → NaN
  EOR_Total : 1 zéro(s) → NaN
  EP_Looker : 2 zéro(s) → NaN


In [7]:
df['jour_semaine']  = df['Date'].dt.dayofweek
df['mois']          = df['Date'].dt.month
df['trimestre']     = df['Date'].dt.quarter
df['nom_jour']      = df['Date'].dt.day_name()
df['semaine_annee'] = df['Date'].dt.isocalendar().week.astype(int)

df['KPI_m3_veh']        = np.where(df['TCM'] >= TCM_MIN_PROD,
                                    df['E. Appro Looker'] / df['TCM'], np.nan)
df['ratio_EOR_ED']       = df['EOR_Total'] / df['ED_Total'].replace(0, np.nan)
df['taux_EP']            = df['EP_Looker'] / df['E. Appro Looker'].replace(0, np.nan)
df['taux_EI']            = df['EI_Looker'] / df['E. Appro Looker'].replace(0, np.nan)
df['anomalie_kpi']       = np.where(
    (df['TCM'] >= TCM_MIN_PROD) & (df['KPI_m3_veh'] > SEUIL_OBJECTIF * 1.15), 1, 0)
df['anomalie_recyclage'] = np.where(df['ratio_EOR_ED'] < 0.10, 1, 0)

df['KPI_MA7']   = df['KPI_m3_veh'].rolling(7,  min_periods=3).mean()
df['KPI_MA30']  = df['KPI_m3_veh'].rolling(30, min_periods=7).mean()
df['conso_MA7'] = df['E. Appro Looker'].rolling(7, min_periods=3).mean()

print(f"Features créées : {len(df.columns)} colonnes")
df[['Date','TCM','KPI_m3_veh','ratio_EOR_ED','anomalie_kpi']].head(10)

Features créées : 64 colonnes


,Date,TCM,KPI_m3_veh,ratio_EOR_ED,anomalie_kpi
0,2026-01-01,0.0,NaN,0.195556,0
1,2026-01-02,0.0,NaN,0.272727,0
2,2026-01-03,0.0,NaN,0.256410,0
3,2026-01-04,0.0,NaN,0.830000,0
4,2026-01-05,45.0,NaN,0.116725,0
5,2026-01-06,691.0,1.123010,0.130952,0
6,2026-01-07,738.0,1.096206,0.052126,0
7,2026-01-08,910.0,1.000000,0.140187,0
8,2026-01-09,742.0,1.258760,0.140870,0
9,2026-01-10,1161.0,1.141258,0.148208,0


In [8]:
df.to_csv(FICHIER_SORTIE, index=False)
print(f"Dataset sauvegardé : {FICHIER_SORTIE}")
print(f"Lignes : {len(df)} | Colonnes : {len(df.columns)}")

Dataset sauvegardé : ../outputs/dataset_eau_propre.csv
Lignes : 112 | Colonnes : 64


In [9]:
df_prod = df[df['TCM'] >= TCM_MIN_PROD]
print("=== RÉSUMÉ ETL ===")
print(f"Jours production  : {len(df_prod)}")
print(f"Jours arrêt       : {(df['is_arret']==1).sum()}")
print(f"KPI moyen         : {df_prod['KPI_m3_veh'].mean():.3f} m³/véh")
print(f"Anomalies KPI     : {df['anomalie_kpi'].sum()}")
print(f"Anomalies recyclage: {df['anomalie_recyclage'].sum()}")

=== RÉSUMÉ ETL ===
Jours production  : 86
Jours arrêt       : 26
KPI moyen         : 1.182 m³/véh
Anomalies KPI     : 15
Anomalies recyclage: 16
